In [ ]:
pip install -u datasets huggingface_hub fsspec


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [ ]:
pip install peft
# peft = > paremter efecient fine tuing

In [ ]:
pip install accelerate -u


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name , trust_remote_code = True)
#

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [ ]:
def generate_text(model, tokenizer, prompt_text, max_tokens):
    prompt_text = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(
        input_ids=prompt_text["input_ids"],
        attention_mask=prompt_text["attention_mask"],
        max_length=max_tokens,
        repetition_penalty=1.5,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [ ]:
initial_output = generate_text(model, tokenizer, "I want you to act as a logistician. ", 100)
print("Initial model output:", initial_output)

Initial model output: ['I want you to act as a logistician.  You will be able to:  Analyze the data']


In [ ]:
from datasets import load_dataset

dataset_prompt = "fka/awesome-chatgpt-prompts"


data_prompt = load_dataset(dataset_prompt)
data_prompt = data_prompt.map(lambda x: tokenizer(x["prompt"]), batched=True)
train_prompts = data_prompt["train"].select(range(30))

README.md: 0.00B [00:00, ?B/s]

prompts.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/759 [00:00<?, ? examples/s]

Map:   0%|          | 0/759 [00:00<?, ? examples/s]

In [ ]:
print(data_prompt['train'][:5])

{'act': ['Ethereum Developer', 'Linux Terminal', 'English Translator and Improver', 'Job Interviewer', 'JavaScript Console'], 'prompt': ['Imagine you are an experienced Ethereum developer tasked with creating a smart contract for a blockchain messenger. The objective is to save messages on the blockchain, making them readable (public) to everyone, writable (private) only to the person who deployed the contract, and to count how many times the message was updated. Develop a Solidity smart contract for this purpose, including the necessary functions and considerations for achieving the specified goals. Please provide the code and any relevant explanations to ensure a clear understanding of the implementation.', 'I want you to act as a linux terminal. I will type commands and you will reply with what the terminal should show. I want you to only reply with the terminal output inside one unique code block, and nothing else. do not write explanations. do not type commands unless I instruct y

In [ ]:
import pandas as pd

df = pd.DataFrame(data_prompt['train'][:])

df.head()
# the result will have these columns :
# act => يشير الي الفعل المراد القيام به بناء ع النص الموجود بالعمود prompt
# prompt => يحتوي ع النص الذي يتم استخامه بتوليد النصوص وهية عبارة عن موجهات بدنا ندرب النموذج عليها منشان يحسن من اداءه
#input_ids =>   على تم يحتوي  تمثيل ال tokens لل prompts الموجودة في العمود الي قبل وال tokens  هية عبارة عن المدخلات الي بتمثل النصوص كارقام
# attention_mask=> بنستخدهه منشان نعرف املنوذج وين لازم يركز اثناء التدريب او التتنبأ وبيكون عبارة عن صفر وواحد الواحد بيرمز للتوكنز المهمة والصفر للغير مهمة

,act,prompt,for_devs,type,contributor,input_ids,attention_mask
0,Ethereum Developer,Imagine you are an experienced Ethereum develo...,True,TEXT,ameya-2003,"[186402, 1152, 1306, 660, 72560, 28857, 167625...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,Linux Terminal,I want you to act as a linux terminal. I will ...,True,TEXT,f,"[44, 4026, 1152, 427, 1769, 661, 267, 104105, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,English Translator and Improver,"I want you to act as an English translator, sp...",False,TEXT,f,"[44, 4026, 1152, 427, 1769, 661, 660, 7165, 24...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,Job Interviewer,I want you to act as an interviewer. I will be...,False,TEXT,"f,iltekin","[44, 4026, 1152, 427, 1769, 661, 660, 33322, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,JavaScript Console,I want you to act as a javascript console. I w...,True,TEXT,omerimzali,"[44, 4026, 1152, 427, 1769, 661, 267, 49760, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
from peft import get_peft_model, PromptTuningConfig, TaskType, PromptTuningInit

tuning_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=4,
    tokenizer_name_or_path=model_name
)

peft_model = get_peft_model(model, tuning_config)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    use_cpu=True,
    output_dir = "./",
    auto_find_batch_size=True,
    learning_rate= 0.005,
    num_train_epochs=5,
    report_to="none",
    )

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_prompts,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
    )
trainer.train()

Step,Training Loss


TrainOutput(global_step=20, training_loss=3.5695144653320314, metrics={'train_runtime': 1293.6106, 'train_samples_per_second': 0.116, 'train_steps_per_second': 0.015, 'total_flos': 31380138393600.0, 'train_loss': 3.5695144653320314, 'epoch': 5.0})

In [ ]:
tuned_output = generate_text(trainer.model, tokenizer, "I want you to act as a logistician. ", 100)
print("Tuned model output:", tuned_output)

Tuned model output: ['I want you to act as a logistician.  You will need the following information:  A computer']
